# RoundRobinGroupChat
RoundRobinGroupChat は、すべてのエージェントが同じコンテキストを共有し、ラウンドロビン方式で順番に返信するシンプルなながらも効果的なチーム構成です。各エージェントは、自分の番が回ってきた際に、他のすべてのエージェントに返信をブロードキャストし、チーム全体が一致したコンテキストを維持します。


In [ ]:
#!pip install -U agent-framework

In [ ]:
import logging  
from typing import Any, List  
import os  
from dotenv import load_dotenv  
from agent_framework.azure import AzureOpenAIChatClient
from agent_framework import Agent, MCPStreamableHTTPTool
from agent_framework.orchestrations import GroupChatBuilder, GroupChatState

load_dotenv(override=True)

## OpenTelemetry によるトレーサーのセット
マルチエージェントのデバッグには OpenTelemetry によるトレーサーを利用すると便利です。
この環境の Agent Framework では `setup_observability` は提供されていないため、
OpenTelemetry の `TracerProvider`（Exporter/Processor 含む）を手動でセットし、`enable_instrumentation()` で計測を有効化します。
ここではトレースの送信先として OTLP gRPC（例: Jaeger / OpenTelemetry Collector の `4317`）を使います。

Jaeger UI: [http://localhost:16686](http://localhost:16686)

In [ ]:
import os

from agent_framework.observability import configure_otel_providers, get_tracer

service_name = "agent_framework"
otlp_endpoint = os.getenv("OTEL_EXPORTER_OTLP_ENDPOINT", "http://localhost:4317")

# 環境変数ベースで設定（Agent Framework は標準の OTEL_* を読む）
os.environ.setdefault("OTEL_SERVICE_NAME", service_name)
os.environ.setdefault("OTEL_EXPORTER_OTLP_ENDPOINT", otlp_endpoint)
os.environ.setdefault("OTEL_EXPORTER_OTLP_PROTOCOL", "grpc")
os.environ.setdefault("ENABLE_INSTRUMENTATION", "true")
os.environ.setdefault("OTEL_METRICS_EXPORTER", "none")  # Metrics は無効化（必要に応じて変更）

# enable_sensitive_data=True を指定して機微データ(OpenAI の IN/OUT データ)の収集を有効化
configure_otel_providers(enable_sensitive_data=True)

tracer = get_tracer()
print(f"Observability configured: service={service_name} otlp={otlp_endpoint}")

## Microsoft Foundry での AI エージェントのトレースと監視

In [ ]:
# # パッケージのインストール
# #pip install azure-ai-projects azure-identity azure-monitor-opentelemetry opentelemetry-sdk
# os.environ["AZURE_TRACING_GEN_AI_CONTENT_RECORDING_ENABLED"] = "true" # False by default

# PROJECT_ENDPOINT = os.environ["PROJECT_ENDPOINT"]

# # コード例
# from azure.ai.projects import AIProjectClient
# from azure.identity import DefaultAzureCredential
# from azure.monitor.opentelemetry import configure_azure_monitor

# # プロジェクトクライアントの作成
# project_client = AIProjectClient(
#     # credential=DefaultAzureCredential(),
#     credential=DefaultAzureCredential(),
#     endpoint=os.environ["PROJECT_ENDPOINT"],
# )

# # Application Insights の接続文字列を取得してトレースを有効化
# connection_string = project_client.telemetry.get_application_insights_connection_string()
# configure_azure_monitor(connection_string=connection_string)

# # # トレーサーを取得
# # from opentelemetry import trace
# # tracer = trace.get_tracer(__name__)

In [ ]:
mcp_server_uri = os.getenv("MCP_SERVER_URI")
azure_openai_key = os.getenv("AZURE_OPENAI_API_KEY")
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")

# === 認証方式の選択 ===
# True: APIキー認証, False: DefaultAzureCredential 認証
USE_KEY_AUTH = os.getenv("USE_KEY_AUTH", "False").lower() in ("true", "1", "t")

if USE_KEY_AUTH:
    auth_kwargs = dict(api_key=azure_openai_key)
    print("🔑 認証方式: APIキー認証")
else:
    from azure.identity.aio import DefaultAzureCredential
    auth_kwargs = dict(credential=DefaultAzureCredential())
    # フレームワークが環境変数の AZURE_OPENAI_API_KEY を自動読み込みして
    # credential より優先してしまうため、明示的に削除する
    os.environ.pop("AZURE_OPENAI_API_KEY", None)
    print("🔐 認証方式: DefaultAzureCredential")

print(f"MCP Server URI: {mcp_server_uri}")
print(f"Azure OpenAI Endpoint: {azure_openai_endpoint}, Deployment: {azure_deployment}")

## Tools の定義(MCP Streamable HTTP)

In [ ]:
# MCP Streamable HTTP ツールを作成（初回接続が重い場合があるので timeout を長めにする）
mcp_tool = MCPStreamableHTTPTool(
    name="mcp_tools",
    url=mcp_server_uri,
    description="ゲームショップの製品、注文、在庫、ユーザー情報、Twitter分析データを取得するためのツールです。",
    timeout=120,
)


# Azure OpenAI クライアントの作成
chat_client = AzureOpenAIChatClient(
    **auth_kwargs,
    endpoint=azure_openai_endpoint,
    api_version=api_version,
    deployment_name=azure_deployment,
)

print(f"MCP Tool: {mcp_tool}")

## マネージャー/コーディネーターエージェント
グループチャットのオーケストレーション用にマネージャー/コーディネーターエージェントを設定します。
マネージャーは、会話の状態とタスク要件に基づいて次に発言する参加者を選択し、参加者を調整します。マネージャーは完全なワークフロー参加者であり、すべてのエージェントインフラストラクチャ（ツール、コンテキスト、可観測性）にアクセスできます。
マネージャーエージェントは、発言者選択の決定を伝達するために、ManagerSelectionResponseと互換性のある構造化された出力を生成する必要があります。確実な解析のためにresponse_formatを使用してください。GroupChatBuilder


## エージェント定義

In [ ]:
# 話者選択のための構造化出力を持つコーディネーターエージェントを作成
# 注意: orchestrator として Agent を使う場合、話者選択のための structured output は
# GroupChat がエージェント呼び出し時に内部的に設定します（ユーザー側で response_format を設定する必要はありません）。
coordinator = Agent(
    name="Coordinator",
    description="話者を選択してマルチエージェントの協調作業を調整します",
    instructions="""
あなたはユーザーのタスクを解決するためのチーム会話を調整します。

会話の履歴を確認し、次に発言する参加者を選択してください。

ガイドライン:
- まず Researcher に情報収集をさせます
- 次に Writer に最終的な回答をまとめさせます
- 両者が有意義な貢献をした後にのみ終了します
- 必要に応じて複数ラウンドの情報収集を許可します
""",
    client=chat_client,
 )

researcher = Agent(
    name="Researcher",
    description="関連する背景情報を収集します",
    instructions="チームメイトが質問に答えるのに役立つ簡潔な事実を収集してください。",
    tools=[mcp_tool],
    client=chat_client,
 )

writer = Agent(
    name="Writer",
    description="収集された情報から洗練された回答を合成します",
    instructions="提供されたメモを使用して、明確で構造化された回答を関西弁で作成してください。",
    client=chat_client,
 )

## 停止条件
現行の Agent Framework では、speaker selector（selection_func）で `None` を返して終了…という形ではありません。
終了は次のいずれかで制御します:
- `.with_termination_condition(...)`（会話履歴を見て終了判定）
- `.with_max_rounds(...)`（ラウンド上限）

In [ ]:
workflow = (
    GroupChatBuilder(
        participants=[researcher, writer],
        orchestrator_agent=coordinator,
        # 終了条件: ASSISTANT ロール(エージェント)のメッセージが4つ以上蓄積されたら会話を終了
        # つまり、Researcher と Writer がそれぞれ1回ずつ応答した時点で終了（オーケストレーターの判断で早期終了も可）
        termination_condition=lambda messages: sum(1 for msg in messages if msg.role == "assistant") >= 4,
        # ★ 参加者エージェントのストリーミングイベントをコンシューマーに通すために必須
        # False(デフォルト)だとオーケストレーターの output のみが yield され、
        # 参加者の AgentResponseUpdate(トークンチャンク)はフィルタされてしまう
        intermediate_outputs=True,
    )
    .build()
 )

### このワークフロー例で確認していること（Coordinator / Researcher / Writer）

この例は「タスクを分業する複数エージェント」が、オーケストレーター（Coordinator）主導で最低限成立するかのスモークテストです。主に次を確認しています。

- **オーケストレーションが効くか**: `.with_orchestrator(agent=coordinator)` により、Coordinator が次に誰（Researcher/Writer）を動かすかを判断できること
- **ツール利用を含む役割分担ができるか**: Researcher だけに `tools=[mcp_tool]` を渡し、情報取得（ツール呼び出し）→Writer が統合、という分業が成立すること
- **共有コンテキストの維持**: Researcher の出力が会話履歴として残り、Writer がそれを参照して最終回答を組み立てられること
- **終了制御ができるか**: `with_termination_condition(...)` で「所定の応答数になったら止まる」こと（無限に回らない）

つまり、外部データの正確性よりも、**マルチエージェントの制御・文脈共有・停止条件**の配線確認が目的です。

## ユーティリティコード
- `run_stream_pretty` — ストリーミング出力をリアルタイム表示するヘルパー
- `visualize_workflow` — ワークフローSVG可視化

### `intermediate_outputs=True` が必要な理由

`GroupChatBuilder` はデフォルトで `intermediate_outputs=False` です。
この場合 `output_executors` がオーケストレーターのみに限定され、
参加者エージェントの `AgentResponseUpdate`（トークンチャンク）は
`Workflow._should_yield_output_event()` でフィルタされ、
コンシューマー（`async for event in workflow.run(stream=True)`）に到達しません。

**結果**: ストリーミングで `print` しているはずが一切表示されず、
最後に `list[Message]`（最終会話ログ）だけが返る状態になります。

`intermediate_outputs=True` を指定すると `output_executors=None` となり、
全エグゼキュータの output イベントが yield されるため、
参加者のトークン単位のストリーミングが正しく動作します。

In [ ]:
from __future__ import annotations
import sys
from contextlib import nullcontext
from typing import Optional, cast
from agent_framework import AgentResponse, AgentResponseUpdate, Message, WorkflowEvent

async def run_stream_pretty(
    workflow,
    task: str,
    *,
    tracer=None,
    span_name: str = "RoundRobinGroupChat",
    print_final: bool = True,
) -> list[Message]:
    """
    
    workflow.run(task, stream=True) を実行し、ストリーミング出力を
    Jupyter Notebook 上でリアルタイムに表示するヘルパー。

    前提（GroupChatBuilder 使用時）:
      GroupChatBuilder(..., intermediate_outputs=True) でビルドすること。
      デフォルト(False)ではオーケストレーターの output のみが yield され、
      参加者の AgentResponseUpdate はフィルタされる。

    表示:
      - AgentResponseUpdate → トークンを逐次表示（sys.stdout.write + flush）
      - executor 切替時 → 改行 + 名前ヘッダ
      - 最終 list[Message] → CONVERSATION LOG としてまとめ表示

    注意:
      ★ start_as_current_span() を使用し、ワークフロー内部のスパンを
      このスパンの子として正しくネストさせる。
      _process_events は通常の async 関数であり async generator ではないため、
      コンテキストマネージャと組み合わせても GeneratorExit 問題は発生しない。
    """
    final_conversation: list[Message] = []
    last_executor_id: Optional[str] = None

    def _write(text: str) -> None:
        """Jupyter でも確実にフラッシュする書き込み"""
        sys.stdout.write(text)
        sys.stdout.flush()

    async def _process_events(workflow, task):
        nonlocal final_conversation, last_executor_id
        async for event in workflow.run(task, stream=True):
            if not isinstance(event, WorkflowEvent):
                continue

            # ----- executor 切替通知 -----
            if event.type == "executor_invoked":
                eid = event.executor_id
                if eid != last_executor_id:
                    if last_executor_id is not None:
                        _write("\n")
                    _write(f"🤖 {eid}: ")
                    last_executor_id = eid

            # ----- output イベント -----
            elif event.type == "output":
                data = event.data

                # (1) ストリーミングチャンク: AgentResponseUpdate
                if isinstance(data, AgentResponseUpdate):
                    eid = event.executor_id
                    if eid != last_executor_id:
                        if last_executor_id is not None:
                            _write("\n")
                        _write(f"🤖 {eid}: ")
                        last_executor_id = eid
                    chunk = data.text or ""
                    if chunk:
                        _write(chunk)

                # (2) 非ストリーミング応答: AgentResponse
                elif isinstance(data, AgentResponse):
                    eid = event.executor_id
                    if eid != last_executor_id:
                        if last_executor_id is not None:
                            _write("\n")
                        _write(f"🤖 {eid}: ")
                        last_executor_id = eid
                    text = data.text or ""
                    if text:
                        _write(text)

                # (3) 最終会話ログ: list[Message]
                elif isinstance(data, list):
                    final_conversation = cast(list[Message], data)

    # ★ start_as_current_span() を使用してワークフロー内部スパンを子としてネスト。
    #   _process_events は通常の async 関数なのでコンテキスト競合は発生しない。
    cm = tracer.start_as_current_span(span_name) if tracer else nullcontext()
    with cm:
        await _process_events(workflow, task)

    # ストリーム末尾の改行
    _write("\n")

    if print_final and final_conversation:
        print("\n" + "=" * 80)
        print("CONVERSATION LOG")
        print("=" * 80)
        for msg in final_conversation:
            author = getattr(msg, "author_name", None) or msg.role
            text = msg.text or str(msg)
            print(f"\n[{author}]")
            print(text)
            print("-" * 80)

    return final_conversation

In [ ]:
from agent_framework import WorkflowViz
from IPython.display import SVG, display
import os

def visualize_workflow(workflow, filename="workflow_diagram", show_preview=True):
    """
    ワークフローのグラフ情報を出力し、SVGで保存し、プレビューする関数
    
    Args:
        workflow: 可視化するワークフローオブジェクト
        filename: 保存するファイル名（拡張子なし）
        show_preview: プレビューを表示するかどうか
    
    Returns:
        保存されたSVGファイルのパス
    """
    # WorkflowVizオブジェクトを作成
    viz = WorkflowViz(workflow)
    
    # 3. SVGファイルとして保存
    try:
        svg_path = viz.export(format="svg", filename=filename)
        print("=" * 60)
        print(f"✅ SVGファイルを保存しました: {svg_path}")
        print("=" * 60)
        print()
        
        # 4. SVGをプレビュー表示
        if show_preview and os.path.exists(svg_path):
            display(SVG(filename=svg_path))
        
        return svg_path
        
    except ImportError as e:
        print("❌ エラー: 'graphviz'パッケージがインストールされていません")
        print("インストール方法: pip install agent-framework[viz] --pre")
        print(f"詳細: {e}")
        return None
    except Exception as e:
        print(f"❌ エラーが発生しました: {e}")
        return None

In [ ]:
# 関数を使ってワークフローを可視化
svg_path = visualize_workflow(
    workflow, 
    filename="collaborative_multi_agent_round_robin_workflow",
    show_preview=True
)

In [ ]:
task = "商品ID:1001の商品詳細を調べて、在庫状況と最近のTwitterでの評判を含む説明を書いてください。"

# run_stream をヘルパーで実行（stream表示 + 最終会話の回収）
final_conversation = await run_stream_pretty(workflow, task, tracer=tracer)

In [ ]:
for msg in final_conversation:
    print(msg.to_dict())

## シナリオ1: 東京vs地方ディベート: どこで暮らすのが豊かな人生か

8人の多様な立場のエージェントが「東京か地方か」というテーマについてディスカッションを行います。モデレーターが各話者を戦略的に選択し、自然な対話の流れを作り出します。

## エージェント定義（ディベート参加者）

In [ ]:
# モデレーター: 話者選択を行い、ディスカッションを進行する
moderator = Agent(
    name="Moderator",
    description="次の話者を選択してディスカッションを導く",
    instructions="""あなたは「東京か地方か、どこで暮らすのが豊かな人生か」というテーマについて、議論を導く思慮深いモデレーターです。

参加者は多様な立場と経験を持っています。戦略的に話者を選択してください:
- 対立する意見を建設的にぶつけ合わせる
- すべての視点が公平に聞かれるようにする
- 具体的な経験やデータに基づく議論を促す
- 感情論だけでなく論理的な考察も引き出す
- 現実的な解決策や新しい視点を見出す

次のことができる話者を選択してください:
1. 直前の発言に具体的に反応する
2. 異なる角度から問題を捉え直す
3. 実体験に基づく説得力のある意見を述べる
4. 議論を深化させる本質的な問いを投げかける

次の場合に終了してください:
- 複数の視点が十分に交換された(6-8回以上)
- 主要な論点が多角的に議論された
- 新しい気づきや統合された視点が現れた

final_message では、議論で浮かび上がった主要な論点と多様な価値観を簡潔にまとめてください。
""",
    client=chat_client,
)

# 多様な立場の8人の参加者
tokyo_salaryman = Agent(
    name="TokyoSalaryman",
    description="東京生まれ東京育ちの大企業社員",
    instructions="""あなたは東京で生まれ育ち、大手企業で働く30代の会社員です。
便利さ、キャリア機会、文化的刺激を重視しています。地方の生活は想像しにくいです。

本音で語ってください。自由に:
- 東京での生活の利点を具体的に述べる
- 地方暮らしへの疑問や不安を率直に表現する
- 他の参加者の意見に反論したり質問する
- 簡潔に(2-4文)、でも説得力を持って
""",
    client=chat_client,
)

local_farmer = Agent(
    name="LocalFarmer",
    description="地方で農業を営む40代",
    instructions="""あなたは地方で代々続く農家を継いでいる40代です。
自然、コミュニティ、ゆとりある生活を大切にしています。都会の忙しさは理解できません。

本音で語ってください。自由に:
- 地方での暮らしの豊かさを具体的に語る
- 東京暮らしへの疑問を率直に述べる
- 他の参加者の意見に地方の視点から応答する
- 簡潔に(2-4文)、でも実感を込めて
""",
    client=chat_client,
)

remote_worker = Agent(
    name="RemoteWorker",
    description="地方在住のリモートワーカー",
    instructions="""あなたは東京のIT企業でリモート勤務しながら、地方都市で暮らす30代です。
「いいとこ取り」の働き方を実践中。両方の良さと課題を知っています。

本音で語ってください。自由に:
- 地方リモートワークの現実を語る(良い面も悪い面も)
- 東京と地方のハイブリッドな生き方を提案する
- 実体験に基づいて他者の意見に応答する
- 簡潔に(2-4文)、でもリアルに
""",
    client=chat_client,
)

u_turn_entrepreneur = Agent(
    name="UTurnEntrepreneur",
    description="Uターンして起業した元東京勤務者",
    instructions="""あなたは東京で10年働いた後、地元にUターンして起業した30代後半です。
両方を経験したからこそ見える違いがあります。地方の可能性を信じています。

本音で語ってください。自由に:
- 東京と地方の両方を経験した視点を語る
- Uターンの決断理由と現実を率直に述べる
- 地方でのビジネスの可能性と課題を共有する
- 簡潔に(2-4文)、でも説得力を持って
""",
    client=chat_client,
)

college_student = Agent(
    name="CollegeStudent",
    description="地方出身で東京の大学に通う学生",
    instructions="""あなたは地方出身で、東京の大学に通う20代前半の学生です。
地元に戻るか東京で就職するか悩んでいます。若い世代の葛藤を抱えています。

本音で語ってください。自由に:
- 若者の視点から東京と地方の魅力と不安を語る
- 将来への迷いや希望を率直に表現する
- 先輩世代の意見に質問や反応をする
- 簡潔に(2-4文)、でも素直に
""",
    client=chat_client,
)

local_civil_servant = Agent(
    name="LocalCivilServant",
    description="地方公務員として地域活性化に取り組む職員",
    instructions="""あなたは地方自治体で地域活性化を担当する30代の公務員です。
人口減少や東京一極集中の現実を目の当たりにしています。地方の未来を真剣に考えています。

本音で語ってください。自由に:
- 地方が抱える現実的な課題を具体的に述べる
- 政策的な視点から東京集中の問題を指摘する
- データや事例を交えて議論に貢献する
- 簡潔に(2-4文)、でも現実的に
""",
    client=chat_client,
)

tokyo_real_estate_owner = Agent(
    name="TokyoRealEstateOwner",
    description="都内で不動産賃貸業を営むオーナー",
    instructions="""あなたは都内で複数の賃貸物件を所有する50代のオーナーです。
東京の不動産市場と人の流れを肌で感じています。経済的視点を重視します。

本音で語ってください。自由に:
- 東京の経済的優位性を具体的に語る
- 不動産投資の視点から居住地の価値を論じる
- 市場原理の観点から意見を述べる
- 簡潔に(2-4文)、でも説得力を持って
""",
    client=chat_client,
)

dual_location = Agent(
    name="DualLocation",
    description="東京と地方の二拠点生活を実践する人",
    instructions="""あなたは東京にマンション、地方に古民家を持ち、二拠点生活を実践する40代です。
「どちらか」ではなく「両方」という選択肢を提示できます。新しいライフスタイルを模索中です。

本音で語ってください。自由に:
- 二拠点生活の実際を具体的に語る(コスト、時間、メリット)
- 二者択一ではない第三の選択肢を提案する
- 実践者としての経験を共有する
- 簡潔に(2-4文)、でもリアルに
""",
    client=chat_client,
)

## ワークフロー構築とトピック設定

In [ ]:
# ディベート用のワークフローを構築
debate_workflow = (
    GroupChatBuilder(
        participants=[tokyo_salaryman, local_farmer, remote_worker, u_turn_entrepreneur, 
                      college_student, local_civil_servant, tokyo_real_estate_owner, dual_location],
        orchestrator_agent=moderator,
        # 終了条件: 10回以上の発言(各エージェントが複数回発言できる)
        termination_condition=lambda messages: sum(1 for msg in messages if msg.role == "assistant") >= 10,
        # ★ 参加者のストリーミング出力をリアルタイムに表示するために必須
        intermediate_outputs=True,
    )
    .build()
 )

# ディベートのトピック
debate_topic = "東京か地方か、どこで暮らすのが豊かな人生だと思いますか?"

### このワークフロー例で確認していること（シナリオ1: ディベート）

この例は、参加者が多い状況で「モデレーター（orchestrator）が会話を回す」ことを確認するデモです。主に次を見ています。

- **多数参加者の中からの話者選択**: 8人の参加者から、Moderator が会話の流れに応じて次の話者を選び続けられること
- **同一コンテキスト下での多視点の蓄積**: 各参加者の発言が履歴に蓄積し、次の発言者がそれを踏まえて反応できること
- **終了条件の設計**: 参加者数が多い会話でも、`with_termination_condition(... >= 10)` のような基準で収束させられること
- **「結論のまとめ」を促す設計**: Moderator の `final_message` 指示により、議論の論点をまとめる役割を持たせられること


In [ ]:
# ワークフローの可視化
svg_path = visualize_workflow(
    debate_workflow, 
    filename="philosophical_debate_workflow",
    show_preview=True
)

## ディベートの実行

In [ ]:
# 哲学的ディベートを実行
debate_conversation = await run_stream_pretty(
    debate_workflow, 
    debate_topic, 
    tracer=tracer,
    span_name="PhilosophicalDebate"
)

In [ ]:
for msg in debate_conversation:
    print(msg.to_dict())


## シナリオ2: シンプルなスピーカーセレクター関数を使用したグループチャット

このセクションでは、`with_orchestrator(selection_func=...)` を使用して、純粋な Python 関数で話者選択を制御する方法を示します。
selection 関数には `GroupChatState` が渡され、次に発言させたい参加者名（文字列）を返します。
終了は `with_termination_condition(...)` や `with_max_rounds(...)` で制御します。

### スピーカーセレクター関数の定義

In [ ]:
def select_next_speaker(state: GroupChatState) -> str:
    """加算器と乗算器を交互に選択するシンプルなスピーカーセレクター。

    Args:
        state: GroupChatState（current_round / participants / conversation を含む）

    Returns:
        次の話者の名前（必ず participants に存在する名前を返す）
    """
    # current_round は 0 から始まり、参加者が応答するたびにインクリメントされます。
    # 偶数ラウンド(0,2,4,...)ではAdder、奇数ラウンド(1,3,5,...)ではMultiplierを次の話者として選び、2人を交互に発言させます。
    return "Adder" if (state.current_round % 2 == 0) else "Multiplier"

### エージェント定義（計算機用）

In [ ]:
# 加算エージェント
adder = Agent(
    name="Adder",
    description="加算器（+100演算）",
    instructions="""あなたは計算チェーンの一員です。

重要: 会話履歴を必ず確認して、最後に発言されたメッセージから数値を抽出してください。

あなたの役割:
1. 会話履歴の最後のメッセージを読む
2. そのメッセージに含まれる数値（円）を見つける
3. その数値に100を足す
4. 計算過程を示して結果を出力する

出力形式: 「[前の値] + 100 = [結果]円」

例:
- 前のメッセージ: "100円を持っています" → 100 + 100 = 200円
- 前のメッセージ: "200 × 2 = 400円" → 400 + 100 = 500円
""",
    client=chat_client,
)

# 乗算エージェント
multiplier = Agent(
    name="Multiplier",
    description="乗算器（×2演算）",
    instructions="""あなたは計算チェーンの一員です。

重要: 会話履歴を必ず確認して、最後に発言されたメッセージから数値を抽出してください。

あなたの役割:
1. 会話履歴の最後のメッセージを読む
2. そのメッセージに含まれる数値（円）を見つける
3. その数値に2を掛ける
4. 計算過程を示して結果を出力する

出力形式: 「[前の値] × 2 = [結果]円」

例:
- 前のメッセージ: "100 + 100 = 200円" → 200 × 2 = 400円
- 前のメッセージ: "400 + 100 = 500円" → 500 × 2 = 1000円
""",
    client=chat_client,
)

### ワークフロー構築（計算機）

In [ ]:
# 参加者の指定方法は2つあります:
# 1. リスト形式 - agent.name 属性を使用: participants=[adder, multiplier]
# 2. 辞書形式 - 明示的な名前: participants={"adder": adder, "multiplier": multiplier}
calculator_workflow = (
    GroupChatBuilder(
        participants=[adder, multiplier],  # agent.name を参加者名として使用
        selection_func=select_next_speaker,
        orchestrator_name="Calculator",
        # ★ 参加者のストリーミング出力をリアルタイムに表示するために必須
        intermediate_outputs=True,
    )
    # 念のためラウンド上限も設定（無限ループ防止）
    .with_max_rounds(4)
    # 4回の参加者応答（Adder×2 + Multiplier×2）で終了
    .with_termination_condition(
        lambda messages: sum(1 for msg in messages if msg.role == "assistant") >= 4
    )
    .build()
 )

In [ ]:
# ワークフローの可視化
svg_path = visualize_workflow(
    calculator_workflow, 
    filename="calculator_workflow",
    show_preview=True
)

### 計算機ワークフローの実行

In [ ]:
calculation_task = "200円を持っています"

print(f"\n=== 計算過程: {calculation_task} ===\n")

# 計算を実行
calculation_result = await run_stream_pretty(
    calculator_workflow, 
    calculation_task, 
    tracer=tracer,
    span_name="CalculatorWorkflow"
)

In [ ]:
for msg in calculation_result:
    print(msg.to_dict())


In [ ]:
for msg in calculation_result:
    print(msg.text)

`CalculatorWorkflow` は「計算そのもの」を確認したいのではなく、グループチャットのオーケストレーションが最低限正しく動くかを確かめるための“動作確認用の検証”です。具体的には次を見ています。

- **話者選択が意図どおりか**: `selection_func` が `state.current_round` の偶奇で `"Adder"` / `"Multiplier"` を交互に返し、実際にその順で発言者が切り替わること
- **コンテキスト共有ができているか**: 後続エージェントが会話履歴（直前メッセージ）を読めて、そこから数値を拾って次の計算に使えること
- **終了条件が効くか**: `with_max_rounds(4)` や `with_termination_condition(... >= 4)` により、無限に回らず想定回数で止まること
- **ストリーミング実行の配線確認**: `run_stream_pretty` で executor の切替や最終会話ログ取得が機能すること


## 【参考】with_orchestrator の内部ロジック
`with_orchestrator(...)` は「どの方式で次話者を決めるか」を設定するだけで、**次話者の選定ロジック自体は 2 系統**あります

- **(A) `selection_func=` を渡した場合**  
  Builder は `GroupChatOrchestrator` を使い、各ターンで `GroupChatState(current_round, participants, conversation)` を作って `selection_func(state)` を呼びます。戻り値（話者名）が awaitable なら await し、**participants に存在する名前かをチェック**して次話者にします。  

- **(B) `agent=` を渡した場合（LLMオーケストレーターが選ぶ）**  
  Builder は `AgentBasedGroupChatOrchestrator` を使い、各ターンでオーケストレーターエージェント（**Agent**）を呼び出して「次に誰を話させるか」を **JSON（構造化出力）で返させます**。  
  内部では `instruction` 文字列に「`terminate/reason/next_speaker/final_message` を含むJSONで返せ」「有効な参加者名（case-sensitive）一覧」を埋め込み、それを会話に追加した上で `agent.run(... options={"response_format": AgentOrchestrationOutput})` を実行→返ってきたJSONをパースして `next_speaker` を採用します。  
    - JSON（構造化出力）例:
        ```json
        {"terminate":false,"reason":"Researcher は製品ID 1001の製品詳細、在庫状況、および Twitter での言及データを提供しました。次に、Writer はこの情報を組み込んだ包括的な説明文を作成する必要があります。","next_speaker":"Writer","final_message":null}
        ```


どちらの経路になるかは `build()` 時の解決で分岐していて、`agent=` なら `AgentBasedGroupChatOrchestrator`、`selection_func=` なら `GroupChatOrchestrator` が組み立てられます。  

補足: agent方式では、`next_speaker` が未登録名だと、実際に送信する段で「未登録」扱いで例外になり得ます（参加者チェックは selection_func 方式ほど明示的ではなく、送信処理側で弾かれる形）。